# Raízes de equações

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/03-raizes.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. explicar com um desenho como funcionam a bisseção, a falsa posição, o ponto fixo, Newton e a secante;
> 2. dizer, para cada método, se ele sempre converge e com que **ordem** converge;
> 3. pedir a um agente de IA um resolvedor de equações e **provar** que o código funciona, medindo a ordem de convergência;
> 4. reconhecer os casos em que cada método falha e escolher outro.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "ipywidgets", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

## 1. Problema de engenharia: a perda de carga numa tubulação

Uma adutora de aço comercial com diâmetro $D = 0{,}10$ m e comprimento $L = 100$ m transporta água a $V = 2$ m/s. Quanto de pressão se perde por atrito? Pela equação de Darcy-Weisbach, a perda de carga é

$$
h_f = f \, \frac{L}{D} \, \frac{V^2}{2g},
$$ (eq-darcy)

e tudo depende do **fator de atrito** $f$. Para escoamento turbulento, $f$ é dado pela equação de Colebrook:

$$
\frac{1}{\sqrt{f}} = -2 \log_{10}\!\left( \frac{\varepsilon / D}{3{,}7} + \frac{2{,}51}{\mathrm{Re}\,\sqrt{f}} \right),
$$ (eq-colebrook)

onde $\varepsilon = 0{,}045$ mm é a rugosidade do aço e $\mathrm{Re} = VD/\nu = 2 \times 10^5$ é o número de Reynolds (água, $\nu = 10^{-6}$ m²/s).

O problema: **não existe fórmula que isole $f$** na equação {eq}`eq-colebrook`. A incógnita aparece dos dois lados, uma vez dentro de um logaritmo. Para obter $f$, reescrevemos a equação na forma

$$
g(f) = \frac{1}{\sqrt{f}} + 2 \log_{10}\!\left( \frac{\varepsilon / D}{3{,}7} + \frac{2{,}51}{\mathrm{Re}\,\sqrt{f}} \right) = 0
$$

e procuramos o valor de $f$ que anula $g$. Esse valor é uma **raiz** de $g$. Este capítulo trata exatamente disto: encontrar $x$ tal que $f(x) = 0$ quando não há fórmula para isso.

In [ ]:
# Dados do problema
D, L, V = 0.10, 100.0, 2.0          # m, m, m/s
eps, nu, grav = 0.045e-3, 1.0e-6, 9.81
Re = V * D / nu

def g_colebrook(f):
    return 1 / np.sqrt(f) + 2 * np.log10(eps / (3.7 * D) + 2.51 / (Re * np.sqrt(f)))

f_vals = np.linspace(0.008, 0.06, 400)
fig = go.Figure()
fig.add_hline(y=0, line=dict(color="#444444", width=1))
fig.add_trace(go.Scatter(x=f_vals, y=g_colebrook(f_vals), name="g(f)", line=dict(color=CORES["exata"])))
fig.update_xaxes(title="fator de atrito f (adimensional)")
fig.update_yaxes(title="g(f)")
fig.update_layout(title="A raiz de g(f) é o fator de atrito procurado", showlegend=False)
mostrar(fig)

**O que notar:** a curva cruza o zero uma única vez, perto de $f \approx 0{,}02$. Pelo gráfico conseguimos uma estimativa grosseira. Os métodos deste capítulo transformam essa estimativa num número com quantos dígitos quisermos. Voltaremos a este problema na seção 6.

## 2. A ideia em uma imagem

Há duas famílias de métodos, e a diferença entre elas cabe num desenho.

- **Métodos intervalares** (bisseção, falsa posição) partem de um intervalo $[a, b]$ em que $f$ **troca de sinal**. Se $f$ é contínua, a raiz está lá dentro. A cada passo o intervalo encolhe, e a raiz nunca escapa. Esses métodos são lentos, mas **sempre convergem**.
- **Métodos abertos** (ponto fixo, Newton, secante) partem de um ou dois palpites e usam a forma local da função para "saltar" em direção à raiz. São rápidos, mas **podem divergir**.

In [ ]:
def f_ex(x):
    return x**2 - 2

def df_ex(x):
    return 2 * x

r_bis = raizes.bissecao(f_ex, 0.0, 2.0, max_iter=5)
r_new = raizes.newton(f_ex, df_ex, 2.0, max_iter=3)

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                    subplot_titles=["Bisseção: o intervalo encolhe pela metade",
                                    "Newton: a tangente aponta para a raiz"])
# Painel da bisseção: um segmento por iteração
a, b = 0.0, 2.0
for k in range(5):
    m = (a + b) / 2
    fig.add_trace(go.Scatter(x=[a, b], y=[k, k], mode="lines+markers",
                             line=dict(color=CORES["aproximacao"], width=6), marker=dict(size=6),
                             showlegend=False, hovertemplate=f"passo {k}: [%{{x:.4f}}]<extra></extra>"), 1, 1)
    fig.add_trace(go.Scatter(x=[m], y=[k], mode="markers", marker=dict(color=CORES["destaque"], size=10, symbol="line-ns-open", line=dict(width=3)),
                             showlegend=False, hoverinfo="skip"), 1, 1)
    if f_ex(a) * f_ex(m) < 0:
        b = m
    else:
        a = m
fig.add_vline(x=np.sqrt(2), line=dict(color=CORES["exata"], dash="dash", width=1), row=1, col=1)
fig.update_yaxes(title="passo", autorange="reversed", dtick=1, row=1, col=1)
fig.update_xaxes(title="x", range=[-0.05, 2.05], row=1, col=1)

# Painel de Newton
xs = np.linspace(0.0, 2.3, 300)
fig.add_trace(go.Scatter(x=xs, y=f_ex(xs), line=dict(color=CORES["exata"]), showlegend=False), 1, 2)
fig.add_hline(y=0, line=dict(color="#444444", width=1), row=1, col=2)
for xk, xk1 in zip(r_new.historico[:-1], r_new.historico[1:]):
    fig.add_trace(go.Scatter(x=[xk, xk1], y=[f_ex(xk), 0], mode="lines",
                             line=dict(color=CORES["aproximacao"], dash="dot"), showlegend=False), 1, 2)
    fig.add_trace(go.Scatter(x=[xk, xk], y=[0, f_ex(xk)], mode="lines",
                             line=dict(color=CORES["referencia"], width=1), showlegend=False), 1, 2)
fig.add_trace(go.Scatter(x=r_new.historico, y=[0] * len(r_new.historico), mode="markers",
                         marker=dict(color=CORES["destaque"], size=9), showlegend=False), 1, 2)
fig.update_xaxes(title="x", range=[0, 2.3], row=1, col=2)
fig.update_yaxes(title="f(x) = x² − 2", range=[-2.5, 3.5], row=1, col=2)
fig.update_layout(width=900, height=400, margin=dict(t=70))
mostrar(fig)

**O que notar:** depois de 5 passos a bisseção ainda tem um intervalo de largura $2/2^5 = 0{,}0625$. Com 3 tangentes, Newton já está a menos de $10^{-5}$ da raiz $\sqrt{2}$.

## 3. Formulação matemática

### 3.1 Como medir a velocidade de um método: a ordem de convergência

Seja $r$ a raiz e $e_k = x_k - r$ o erro na iteração $k$. Dizemos que o método converge com **ordem** $q$ quando, perto da raiz,

$$
|e_{k+1}| \approx C \, |e_k|^{q}.
$$ (eq-ordem)

- $q = 1$ (**linear**): o erro é multiplicado por uma constante $C < 1$ a cada passo. Ganha-se um número fixo de dígitos por iteração.
- $q = 2$ (**quadrática**): o número de dígitos corretos **dobra** a cada iteração.

Tomando o logaritmo de {eq}`eq-ordem`, obtemos $\log|e_{k+1}| \approx \log C + q \log|e_k|$. Isso é uma reta de inclinação $q$. Esse é o teste que usaremos para **medir** a ordem de um código: desenhar $\log|e_{k+1}|$ contra $\log|e_k|$ e medir a inclinação. A função `ordem_iterativa` do pacote do livro faz esse ajuste.

### 3.2 Bisseção

Se $f$ é contínua em $[a, b]$ e $f(a)\,f(b) < 0$, o **teorema do valor intermediário** garante uma raiz no intervalo. A bisseção toma o ponto médio $m = (a+b)/2$ e fica com a metade em que o sinal ainda troca.

Depois de $n$ passos, a raiz está num intervalo de largura $(b-a)/2^n$, logo o erro do ponto médio satisfaz

$$
|e_n| \le \frac{b - a}{2^{n+1}}.
$$

Isso dá algo raro em métodos numéricos: o **número de iterações sabido de antemão**. Para garantir erro menor que $\delta$, bastam

$$
n \ge \log_2\!\left(\frac{b-a}{\delta}\right) - 1
$$

passos. O erro cai pela metade a cada passo, então a convergência é linear com $C = 1/2$.

### 3.3 Falsa posição

A bisseção ignora os **valores** de $f$ nas pontas. A falsa posição traça a reta (corda) entre $(a, f(a))$ e $(b, f(b))$ e usa o ponto onde ela corta o eixo:

$$
x = b - f(b)\,\frac{a - b}{f(a) - f(b)}.
$$

Em seguida, mantém o subintervalo em que o sinal troca, como na bisseção. Costuma ser mais rápida, mas tem uma armadilha que veremos na seção 7: se a função é muito curva, uma das pontas fica **presa** e a convergência fica muito lenta.

### 3.4 Iteração de ponto fixo

Reescrevemos $f(x) = 0$ na forma $x = g(x)$ e iteramos $x_{k+1} = g(x_k)$. Por exemplo, $e^{-x} - x = 0$ vira $x = e^{-x}$.

Por que funciona? Se $r = g(r)$, então pelo teorema do valor médio

$$
e_{k+1} = x_{k+1} - r = g(x_k) - g(r) = g'(\xi_k)\,(x_k - r) = g'(\xi_k)\, e_k,
$$

com $\xi_k$ entre $x_k$ e $r$. Logo o erro é multiplicado por aproximadamente $g'(r)$ a cada passo:

- se $|g'(r)| < 1$, o método **converge**, linearmente, com $C = |g'(r)|$;
- se $|g'(r)| > 1$, o método **diverge**, por melhor que seja o palpite inicial.

A mesma equação admite muitas formas $x = g(x)$, e umas convergem enquanto outras não. A seção 5 mostra isso no **diagrama de teia de aranha**.

### 3.5 Newton-Raphson

Expandindo $f$ em série de Taylor em torno do palpite atual $x_k$,

$$
f(x) \approx f(x_k) + f'(x_k)\,(x - x_k).
$$

O lado direito é a **reta tangente**. Igualando-o a zero e chamando a solução de $x_{k+1}$:

$$
x_{k+1} = x_k - \frac{f(x_k)}{f'(x_k)}.
$$ (eq-newton)

**Por que a convergência é quadrática?** Expandindo agora em torno da raiz $r$ até a segunda ordem, com $f(r) = 0$:

$$
f(x_k) = f'(r)\, e_k + \tfrac{1}{2} f''(r)\, e_k^2 + \dots, \qquad
f'(x_k) = f'(r) + f''(r)\, e_k + \dots
$$

Substituindo em {eq}`eq-newton` e subtraindo $r$ dos dois lados:

$$
e_{k+1} = e_k - \frac{f'(r)\, e_k + \tfrac12 f''(r)\, e_k^2}{f'(r) + f''(r)\, e_k}
\;\approx\; \frac{f''(r)}{2 f'(r)} \, e_k^2 .
$$

O erro novo é proporcional ao **quadrado** do erro anterior: $q = 2$ e $C = |f''(r) / 2f'(r)|$. Essa dedução vale só se $f'(r) \neq 0$. Quando $f'(r) = 0$, a raiz é **múltipla**, e a convergência cai para linear (seção 7).

### 3.6 Secante

Quando $f'$ é difícil ou cara de calcular, ela pode ser trocada pela inclinação da reta que passa pelos dois últimos pontos:

$$
x_{k+1} = x_k - f(x_k)\, \frac{x_k - x_{k-1}}{f(x_k) - f(x_{k-1})}.
$$

A fórmula é igual à da falsa posição. A diferença é que a secante sempre descarta o ponto **mais antigo**, mesmo que a raiz deixe de estar entre os dois. Ela perde a garantia de convergência, mas ganha velocidade. Pode-se mostrar que a ordem da secante é a razão áurea, $q = (1 + \sqrt 5)/2 \approx 1{,}618$. Como cada passo custa só uma avaliação de $f$, contra $f$ e $f'$ em Newton, a secante muitas vezes é mais rápida em tempo de computador.

## 4. À mão

Faça no papel, com 6 casas decimais, as primeiras iterações para $f(x) = x^2 - 2$. Depois confira com a célula abaixo.

**Bisseção** em $[1, 2]$:

| $k$ | $a$ | $b$ | $m$ | $f(m)$ | novo intervalo |
| --- | --- | --- | --- | --- | --- |
| 1 | 1 | 2 | 1,5 | 0,25 | $[1;\ 1{,}5]$ |
| 2 | 1 | 1,5 | 1,25 | −0,4375 | $[1{,}25;\ 1{,}5]$ |
| 3 | 1,25 | 1,5 | 1,375 | −0,109375 | $[1{,}375;\ 1{,}5]$ |

**Newton** com $x_0 = 1$, usando $x_{k+1} = x_k - (x_k^2 - 2)/(2x_k)$:

| $k$ | $x_k$ | erro $\lvert x_k - \sqrt2 \rvert$ |
| --- | --- | --- |
| 0 | 1 | 0,414214 |
| 1 | 1,5 | 0,085786 |
| 2 | 1,416667 | 0,002453 |
| 3 | 1,414216 | 0,000002 |

Repare na coluna de erro de Newton: 0,08 → 0,002 → 0,000002. O expoente dobra a cada passo. Essa é a convergência quadrática da seção 3.5, vista nos números.

In [ ]:
r = raizes.newton(f_ex, df_ex, 1.0, max_iter=4)
for k, xk in enumerate(r.historico):
    print(f"k = {k}   x = {xk:.6f}   erro = {abs(xk - np.sqrt(2)):.2e}")

## 5. Explore

Os controles desta seção precisam do Python rodando. Use o botão de executar no topo da página ou abra o capítulo no Colab. Logo abaixo de cada controle há uma versão estática com alguns valores fixos, que é a que aparece no PDF.

### 5.1 Ponto fixo: a teia de aranha

No diagrama, a iteração $x_{k+1} = g(x_k)$ vira um caminho em degraus entre a curva $y = g(x)$ e a reta $y = x$. A raiz é o cruzamento das duas. Escolha a forma de $g$ e o palpite inicial.

In [ ]:
FORMAS_G = {
    "g(x) = exp(−x)":        (lambda x: np.exp(-x), (0.0, 1.2)),
    "g(x) = −ln(x)":         (lambda x: -np.log(x), (0.05, 1.2)),
    "g(x) = (x + exp(−x))/2": (lambda x: (x + np.exp(-x)) / 2, (0.0, 1.2)),
}
RAIZ_EXP = 0.5671432904097838   # raiz de exp(−x) − x

def figura_teia(nome, x0=0.2, passos=8, titulo=None):
    g, (lo, hi) = FORMAS_G[nome]
    xs = np.linspace(lo, hi, 300)
    with np.errstate(all="ignore"):
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=xs, y=g(xs), mode="lines", name=nome, line=dict(color=CORES["exata"])))
        fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", name="y = x", line=dict(color=CORES["referencia"], dash="dash")))
        px, py, x = [x0], [0.0], x0
        for _ in range(passos):
            y = g(x)
            if not np.isfinite(y) or abs(y) > 10:
                break
            px += [x, y]; py += [y, y]
            x = y
    fig.add_trace(go.Scatter(x=px, y=py, mode="lines+markers", name="iterações",
                             line=dict(color=CORES["aproximacao"], width=1.5), marker=dict(size=5)))
    fig.add_trace(go.Scatter(x=[RAIZ_EXP], y=[RAIZ_EXP], mode="markers", name="raiz",
                             marker=dict(color=CORES["destaque"], size=11)))
    fig.update_xaxes(title="x", range=[lo, hi])
    fig.update_yaxes(title="y", range=[lo, hi], scaleanchor="x")
    fig.update_layout(title=titulo or nome, width=560, height=520)
    return fig

def explorar_teia(forma="g(x) = exp(−x)", x0=0.2, passos=8):
    mostrar(figura_teia(forma, x0, passos))

widgets.interact(explorar_teia,
                 forma=widgets.Dropdown(options=list(FORMAS_G), description="forma"),
                 x0=widgets.FloatSlider(value=0.2, min=0.1, max=1.1, step=0.05, description="x₀"),
                 passos=widgets.IntSlider(value=8, min=1, max=30, description="passos"));

In [ ]:
# Versão estática: as três formas, 10 passos cada
nomes = list(FORMAS_G)
palpites = [0.2, 0.5, 0.2]
fig = make_subplots(rows=1, cols=3, subplot_titles=nomes, horizontal_spacing=0.07)
for j, nome in enumerate(nomes, start=1):
    for tr in figura_teia(nome, x0=palpites[j - 1], passos=10).data:
        tr.showlegend = False
        fig.add_trace(tr, row=1, col=j)
    fig.update_xaxes(range=[0, 1.2], title="x", row=1, col=j)
    fig.update_yaxes(range=[0, 1.2], row=1, col=j)
fig.update_yaxes(title="y", col=1)
fig.update_layout(width=960, height=380, margin=dict(t=60))
mostrar(fig)

**O que notar:** as três formas têm a **mesma** raiz, $r \approx 0{,}5671$.
- Com $g(x) = e^{-x}$ temos $|g'(r)| = e^{-r} \approx 0{,}57 < 1$, e a teia espirala para dentro.
- Com $g(x) = -\ln x$ temos $|g'(r)| = 1/r \approx 1{,}76 > 1$, e a teia espirala para **fora**.
- A terceira forma é a média das duas primeiras, com $|g'(r)| \approx 0{,}22$. Ela converge bem mais rápido. Escolher bem $g$ é o que faz o ponto fixo funcionar.

### 5.2 Newton: de onde se parte importa

Para $f(z) = z^3 - 1$ no plano complexo há três raízes: $1$ e $e^{\pm 2\pi i/3}$. Cada ponto do plano abaixo foi usado como palpite inicial de Newton. Ele foi colorido segundo a raiz a que o método chegou, e o tom mais claro indica que foram necessárias mais iterações. As fronteiras entre as **bacias de convergência** são fractais: perto delas, uma mudança mínima no palpite leva a uma raiz diferente.

In [ ]:
def bacias_newton(n=300, lim=1.6, max_iter=40):
    x = np.linspace(-lim, lim, n)
    Z = x[None, :] + 1j * x[:, None]
    raizes_c = np.array([1, np.exp(2j * np.pi / 3), np.exp(-2j * np.pi / 3)])
    iters = np.full(Z.shape, max_iter)
    with np.errstate(all="ignore"):
        for k in range(max_iter):
            Z = Z - (Z**3 - 1) / (3 * Z**2)
            perto = np.min(np.abs(Z[..., None] - raizes_c), axis=-1) < 1e-6
            iters = np.where(perto & (iters == max_iter), k, iters)
    qual = np.argmin(np.abs(Z[..., None] - raizes_c), axis=-1)
    return x, qual, iters, raizes_c

x_b, qual, iters, raizes_c = bacias_newton()
# Valor = raiz (0, 1, 2) + fração de iterações: cada raiz tem sua faixa de cor
valor = qual + np.clip(iters / 25, 0, 0.95)
cores_raiz = [CORES["aproximacao"], CORES["secundaria"], CORES["terciaria"]]
escala = []
for i, c in enumerate(cores_raiz):
    escala += [[i / 3, c], [(i + 0.95) / 3, "#FFFFFF"]]
escala[-1][0] = 1.0
fig = go.Figure(go.Heatmap(x=x_b, y=x_b, z=valor, zmin=0, zmax=3, colorscale=escala, showscale=False,
                           hovertemplate="z₀ = %{x:.3f} %{y:+.3f}i<extra></extra>"))
fig.add_trace(go.Scatter(x=raizes_c.real, y=raizes_c.imag, mode="markers",
                         marker=dict(color="black", size=10, symbol="x"), showlegend=False, hoverinfo="skip"))
fig.update_xaxes(title="parte real de z₀", showgrid=False)
fig.update_yaxes(title="parte imaginária de z₀", scaleanchor="x", showgrid=False)
fig.update_layout(title="Bacias de convergência de Newton para z³ − 1", width=600, height=560)
mostrar(fig)

**O que notar:** longe das fronteiras, o palpite leva à raiz mais próxima, como se espera. Sobre os "braços" que saem da origem, as três cores se alternam em escalas cada vez menores. Num problema real, isso quer dizer que um palpite inicial ruim pode levar Newton a uma raiz que **não é a que interessa**.

### 5.3 Newton com palpite variável

In [ ]:
def f_cubica(x):
    return x**3 - 2 * x + 2

def df_cubica(x):
    return 3 * x**2 - 2

def figura_newton(f, df, x0, passos=4, xlim=(-2.5, 2), ylim=(-4, 6), titulo=None):
    r = raizes.newton(f, df, x0, max_iter=passos)
    xs = np.linspace(*xlim, 400)
    fig = go.Figure()
    fig.add_hline(y=0, line=dict(color="#444444", width=1))
    fig.add_trace(go.Scatter(x=xs, y=f(xs), name="f(x)", line=dict(color=CORES["exata"])))
    for k, (xk, xk1) in enumerate(zip(r.historico[:-1], r.historico[1:])):
        fig.add_trace(go.Scatter(x=[xk, xk1], y=[f(xk), 0], mode="lines", line=dict(color=CORES["aproximacao"], dash="dot"),
                                 name="tangente", showlegend=(k == 0)))
        fig.add_trace(go.Scatter(x=[xk, xk], y=[0, f(xk)], mode="lines",
                                 line=dict(color=CORES["referencia"], width=1), showlegend=False))
    fig.add_trace(go.Scatter(x=r.historico, y=[0] * len(r.historico), mode="markers", name="iterações",
                             marker=dict(color=CORES["destaque"], size=9),
                             hovertemplate="x = %{x:.6f}<extra></extra>"))
    fig.update_xaxes(title="x", range=list(xlim))
    fig.update_yaxes(title="f(x)", range=list(ylim))
    fig.update_layout(title=titulo or f"x₀ = {numero(x0)}: {r.raiz:.4f} após {passos} passos".replace(".", ","))
    return fig

def explorar_newton(x0=-1.0, passos=4):
    mostrar(figura_newton(f_cubica, df_cubica, x0, passos))

widgets.interact(explorar_newton,
                 x0=widgets.FloatSlider(value=-1.0, min=-2.4, max=1.9, step=0.05, description="x₀"),
                 passos=widgets.IntSlider(value=4, min=1, max=12, description="passos"));

In [ ]:
# Versão estática: f(x) = x³ − 2x + 2 a partir de três palpites
valores = [-2.0, 0.0, 0.5]
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.05,
                    subplot_titles=[f"x₀ = {numero(v)}" for v in valores])
for j, v in enumerate(valores, start=1):
    for tr in figura_newton(f_cubica, df_cubica, v, passos=6).data:
        tr.showlegend = (j == 1) and tr.showlegend is not False
        fig.add_trace(tr, row=1, col=j)
fig.update_xaxes(range=[-2.5, 2], title="x")
fig.update_yaxes(range=[-4, 6])
fig.update_yaxes(title="f(x) = x³ − 2x + 2", col=1)
fig.update_layout(width=960, height=400, margin=dict(t=60))
mostrar(fig)

**O que notar:** a única raiz real fica perto de $x \approx -1{,}77$. A partir de $x_0 = -2$, Newton chega lá em poucos passos. A partir de $x_0 = 0$, o método fica preso num **ciclo** entre 0 e 1 para sempre. Com $x_0 = 0{,}5$, as tangentes passeiam pelo gráfico antes de, por sorte, chegar à raiz.

## 6. Trabalhando com o agente

Voltemos ao fator de atrito. Em vez de escrever o resolvedor, vamos **pedi-lo** a um agente de IA e, depois, **provar** que o código funciona.

### 6.1 O pedido

Um bom pedido diz o que entra, o que sai, como parar, o que fazer quando algo dá errado e como testar. Compare:

> ❌ *"Escreva um código que resolve a equação de Colebrook."*

> ✅ *"Escreva em Python uma função `fator_atrito(Re, rugosidade_relativa, tol=1e-12)` que resolve a equação de Colebrook pelo método de Newton, usando a variável $x = 1/\sqrt{f}$. Use como palpite inicial a fórmula explícita de Swamee-Jain. Pare quando a variação relativa de $x$ for menor que `tol` ou após 50 iterações. Lance `ValueError` se Re < 4000 (escoamento não turbulento). Devolva $f$ e a lista com todos os $x_k$. Escreva também testes que comparem o resultado com `scipy.optimize.brentq` para Re entre 10⁴ e 10⁸."*

O segundo pedido escolhe o método e a variável, porque em $x = 1/\sqrt f$ a equação fica quase linear e Newton converge muito bem. Ele também define o critério de parada, trata o caso inválido e já pede o teste. **Escrever esse pedido exige entender o método**, e é aqui que o conteúdo das seções 2 a 4 entra.

### 6.2 O código que voltou

Suponha que o agente devolveu o código abaixo. Em $x = 1/\sqrt f$, a equação é $F(x) = x + 2\log_{10}(a + bx) = 0$, com $a = \varepsilon/(3{,}7D)$ e $b = 2{,}51/\mathrm{Re}$.

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
import math

def fator_atrito(Re, rug_rel, tol=1e-12, max_iter=50):
    if Re < 4000:
        raise ValueError("Colebrook vale apenas para escoamento turbulento (Re >= 4000).")
    a, b = rug_rel / 3.7, 2.51 / Re
    F = lambda x: x + 2 * math.log10(a + b * x)
    dF = lambda x: 1 + 2 * b / (a + b * x)
    f0 = 0.25 / math.log10(rug_rel / 3.7 + 5.74 / Re**0.9) ** 2   # Swamee-Jain
    x = 1 / math.sqrt(f0)
    historico = [x]
    for _ in range(max_iter):
        x_novo = x - F(x) / dF(x)
        historico.append(x_novo)
        if abs(x_novo - x) / abs(x_novo) < tol:
            break
        x = x_novo
    return 1 / x_novo**2, historico

f_agente, hist_agente = fator_atrito(Re, eps / D)
print(f"f = {f_agente:.6f}   em {len(hist_agente) - 1} iterações")

O resultado parece razoável. Mas "parece razoável" não é uma verificação. Vamos testar três coisas.

### 6.3 Verificação 1: o resultado está certo?

Comparamos com um método independente, o `brentq` do SciPy, que é intervalar e robusto, e com o resíduo da própria equação.

In [ ]:
from scipy.optimize import brentq

f_ref = brentq(g_colebrook, 0.005, 0.1, xtol=1e-15)
compara(f_agente, f_ref, rtol=1e-10)
print(f"resíduo g(f) = {g_colebrook(f_agente):.2e}")
print(f"perda de carga h_f = {f_agente * L / D * V**2 / (2 * grav):.3f} m")

O valor está certo até a 10ª casa. Um aluno apressado pararia aqui.

### 6.4 Verificação 2: o método se comporta como Newton?

O pedido era **Newton**, que deve convergir com ordem 2 (seção 3.5). Medimos a ordem a partir do histórico, como na seção 3.1, e comparamos com a nossa implementação de referência:

In [ ]:
a_c, b_c = eps / D / 3.7, 2.51 / Re
F = lambda x: x + 2 * np.log10(a_c + b_c * x)
dF = lambda x: 1 + 2 * b_c / ((a_c + b_c * x) * np.log(10))
x_star = 1 / np.sqrt(f_ref)
r_ref = raizes.newton(F, dF, hist_agente[0], tol=1e-14)

erros_agente = [abs(x - x_star) for x in hist_agente]
erros_ref = [abs(x - x_star) for x in r_ref.historico]

fig = go.Figure()
for nome, erros, cor in [("código do agente", erros_agente, CORES["erro"]),
                         ("Newton de referência", erros_ref, CORES["aproximacao"])]:
    e = np.array([v for v in erros if v > 1e-15])
    fig.add_trace(go.Scatter(x=list(range(len(e))), y=e, mode="lines+markers", name=nome, line=dict(color=cor)))
fig.update_xaxes(title="iteração k")
fig.update_yaxes(type="log", title="erro |x_k − x*|", exponentformat="power")
fig.update_layout(title="O código do agente converge, mas devagar demais para ser Newton")
mostrar(fig)

print("ordem medida, agente:    ", numero(ordem_iterativa(hist_agente, x_star), 2))
print("ordem medida, referência:", numero(ordem_iterativa(r_ref.historico, x_star), 2))

**O que notar:** o código do agente converge **linearmente**. O erro cai sempre pelo mesmo fator, e a curva no gráfico é uma reta. Newton correto chega ao limite da precisão da máquina em 2 passos. A ordem medida da referência, cerca de 1,8, fica perto de 2. Com só dois passos até esse limite, há poucos pontos para o ajuste. Há um erro no código, e o resultado final **esconde** esse erro.

Procurando: a derivada de $\log_{10}(u)$ é $u'/(u \ln 10)$, não $u'/u$. O agente esqueceu o fator $\ln 10$ em `dF`. Com a derivada errada, o método ainda converge para a raiz certa, porque o ponto fixo da iteração continua sendo a raiz de $F$. Mas ele deixa de ser Newton, e a convergência quadrática se perde.

> **Lição.** Comparar o resultado final não basta. O teste de **ordem de convergência** detecta um erro que o teste de resultado deixa passar. Num problema maior (milhares de equações, chamadas dentro de um simulador), esse erro custaria muito tempo de computação.

### 6.5 Verificação 3: funciona em toda a faixa de uso?

Um resolvedor de fator de atrito será chamado para muitos valores de Re e de rugosidade. O teste final percorre a faixa toda e compara com o `brentq`:

In [ ]:
def fator_atrito_corrigido(Re, rug_rel, tol=1e-12):
    a, b = rug_rel / 3.7, 2.51 / Re
    F = lambda x: x + 2 * np.log10(a + b * x)
    dF = lambda x: 1 + 2 * b / ((a + b * x) * np.log(10))
    f0 = 0.25 / np.log10(a + 5.74 / Re**0.9) ** 2
    r = raizes.newton(F, dF, 1 / np.sqrt(f0), tol=tol)
    return 1 / r.raiz**2, r.iteracoes

Re_vals = np.logspace(4, 8, 40)
fig = go.Figure()
pior = 0.0
for rr, cor in [(1e-6, CORES["aproximacao"]), (1e-4, CORES["secundaria"]), (1e-2, CORES["terciaria"])]:
    f_n, its = zip(*[fator_atrito_corrigido(R, rr) for R in Re_vals])
    G = lambda f, R=None: 1 / np.sqrt(f) + 2 * np.log10(rr / 3.7 + 2.51 / (R * np.sqrt(f)))
    f_b = [brentq(lambda f: G(f, R), 1e-4, 0.2, xtol=1e-15) for R in Re_vals]
    pior = max(pior, np.max(np.abs(np.array(f_n) - f_b) / np.array(f_b)))
    rotulo = {1e-6: "10⁻⁶", 1e-4: "10⁻⁴", 1e-2: "10⁻²"}[rr]
    fig.add_trace(go.Scatter(x=Re_vals, y=f_n, mode="lines", name=f"ε/D = {rotulo}", line=dict(color=cor),
                             customdata=its, hovertemplate="Re = %{x:.2e}<br>f = %{y:.5f}<br>%{customdata} iterações<extra></extra>"))
fig.update_xaxes(type="log", title="número de Reynolds Re", exponentformat="power")
fig.update_yaxes(type="log", title="fator de atrito f", tickvals=[0.006, 0.008, 0.01, 0.015, 0.02, 0.03, 0.04])
fig.update_layout(title="Diagrama de Moody calculado pelo resolvedor verificado")
mostrar(fig)
print(f"maior diferença relativa para o brentq em 120 casos: {pior:.1e}")

**O que notar:** este é o **diagrama de Moody**, a figura clássica de mecânica dos fluidos, agora calculado pelo nosso resolvedor. A diferença máxima para o `brentq` está na ordem do arredondamento. Passe o mouse para ver quantas iterações cada ponto exigiu: graças ao bom palpite inicial de Swamee-Jain, poucas.

## 7. Quando falha

Todo método tem casos em que falha. Saber reconhecê-los é a parte do trabalho que não se delega ao agente.

### 7.1 Raiz múltipla: Newton fica lento

Em $f(x) = (x-1)^2(x-3)$, a raiz $x = 1$ é **dupla**: ali $f'(1) = 0$. A dedução da seção 3.5 deixa de valer, e Newton converge só linearmente, com $C = 1/2$. Se a multiplicidade $m$ é conhecida, o **Newton modificado** $x_{k+1} = x_k - m\, f(x_k)/f'(x_k)$ recupera a ordem 2.

In [ ]:
f_dupla = lambda x: (x - 1) ** 2 * (x - 3)
df_dupla = lambda x: (x - 1) * (3 * x - 7)
resultados = {
    "Newton (m = 1)": raizes.newton(f_dupla, df_dupla, 0.0, max_iter=60),
    "Newton modificado (m = 2)": raizes.newton(f_dupla, df_dupla, 0.0, m=2),
}
mostrar(grafico_iteracoes(resultados, titulo="Raiz dupla: Newton perde a convergência quadrática"))
for nome, r in resultados.items():
    print(f"{nome:28s} {r.iteracoes:3d} iterações, ordem medida {numero(ordem_iterativa(r.historico, 1.0), 2)}")

### 7.2 Falsa posição com uma ponta presa

Para $f(x) = x^{10} - 1$ em $[0;\ 1{,}3]$, a função é quase plana à esquerda e muito íngreme à direita. A corda sempre corta o eixo perto da ponta esquerda, e a ponta direita fica **presa** em 1,3. Aqui a bisseção vence com folga.

In [ ]:
f10 = lambda x: x**10 - 1
resultados = {
    "bisseção": raizes.bissecao(f10, 0.0, 1.3),
    "falsa posição": raizes.falsa_posicao(f10, 0.0, 1.3, max_iter=500),
}
mostrar(grafico_iteracoes(resultados, titulo="x¹⁰ − 1: a falsa posição perde para a bisseção"))
for nome, r in resultados.items():
    print(f"{nome:14s} {r.iteracoes:4d} iterações")

### 7.3 Ciclos e divergência

Os métodos abertos não têm garantia de convergência. Dois exemplos clássicos:
- $f(x) = x^3 - 2x + 2$ com $x_0 = 0$: Newton alterna entre 0 e 1 para sempre (seção 5.3).
- $f(x) = \arctan x$ com $x_0 = 1{,}5$: cada tangente lança o palpite mais longe, e Newton **diverge**.

In [ ]:
r_ciclo = raizes.newton(f_cubica, df_cubica, 0.0, max_iter=8)
r_atan = raizes.newton(np.arctan, lambda x: 1 / (1 + x**2), 1.5, max_iter=6)
print("x³ − 2x + 2, x₀ = 0:  ", [round(v, 4) for v in r_ciclo.historico])
print("arctan(x),   x₀ = 1,5:", [f"{v:.3g}" for v in r_atan.historico])
print(r_ciclo, r_atan, sep="\n")

A defesa prática é **combinar** as duas famílias. Os métodos híbridos, como o de Brent (`scipy.optimize.brentq`), mantêm um intervalo com troca de sinal, como a bisseção, e tentam passos rápidos, como os da secante e da interpolação. Um passo rápido só é aceito quando cai dentro do intervalo. O resultado tem a garantia da bisseção e, perto da raiz, a velocidade dos métodos abertos. Por isso é o padrão em bibliotecas.

### 7.4 A bisseção também pode enganar

A bisseção exige $f$ **contínua**. Para $f(x) = \tan x$ em $[1;\ 2]$ há troca de sinal, mas não há raiz: há um **polo** em $x = \pi/2$. A bisseção "converge" alegremente para o polo.

In [ ]:
r_tan = raizes.bissecao(np.tan, 1.0, 2.0)
print(r_tan)
print(f"x = {r_tan.raiz:.10f},   π/2 = {np.pi / 2:.10f},   f(x) = {np.tan(r_tan.raiz):.3e}")

**O que notar:** o método diz que convergiu, mas $f(x)$ vale cerca de $10^{10}$ no ponto encontrado. A verificação mais simples de todas, **calcular o resíduo $f(x)$ no resultado**, denuncia o problema. Ela deve ser sempre o primeiro teste, venha o código de onde vier.

## Resumo

| Método | Precisa de | Ordem | Sempre converge? | Quando usar |
| --- | --- | --- | --- | --- |
| Bisseção | intervalo com troca de sinal | 1 ($C = 1/2$) | sim, se $f$ é contínua | quando a robustez importa mais que a velocidade; para obter um bom palpite |
| Falsa posição | intervalo com troca de sinal | 1 | sim, se $f$ é contínua | funções pouco curvas; evitar quando uma ponta pode ficar presa |
| Ponto fixo | uma forma $x = g(x)$ com $\lvert g'(r)\rvert < 1$ | 1 ($C = \lvert g'(r)\rvert$) | não | quando a própria física sugere a forma $x = g(x)$ |
| Newton | $f$ e $f'$, bom palpite | 2 (1 em raiz múltipla) | não | derivada barata e bom palpite inicial |
| Secante | dois palpites | ≈ 1,618 | não | derivada cara ou indisponível |
| Brent (`brentq`) | intervalo com troca de sinal | superlinear perto da raiz | sim | **padrão para uso geral** |

## Exercícios

**Conceituais (sem computador)**

1. Quantas iterações de bisseção são necessárias para obter a raiz de $f(x) = x^3 - x - 1$ em $[1, 2]$ com erro menor que $10^{-8}$? E de Newton, supondo convergência quadrática com $C \approx 1$ a partir de um erro inicial de $0{,}1$?
2. Para $x^2 - 2 = 0$, proponha duas formas $x = g(x)$, uma que converge e outra que diverge. Justifique com $g'(\sqrt 2)$.
3. Mostre que Newton aplicado a $f(x) = x^2 - a$ é o método babilônico $x_{k+1} = \tfrac12(x_k + a/x_k)$.
4. Na seção 6.4, explique por que o código com a derivada errada ainda converge para a raiz certa. Com que fator $C$ você espera que ele convirja?

**Verificação (com agente)**

5. Peça a um agente uma implementação do método da secante. Verifique, pelo gráfico da seção 3.1, que a ordem medida é próxima de 1,618. Entregue o pedido, o código, o gráfico e a ordem medida.
6. Peça ao agente uma função que encontre **todas** as raízes reais de um polinômio num intervalo. Construa um polinômio que faça a função falhar: uma raiz dupla, duas raízes muito próximas ou uma raiz exatamente numa ponta do intervalo. Explique a falha.
7. Peça a um agente um resolvedor para a equação de Kepler $M = E - e\sin E$ (mecânica orbital). Verifique o resultado, a ordem de convergência e o comportamento para excentricidade $e$ próxima de 1.

**Projeto aberto**

8. **Rede de tubulações.** Uma bomba alimenta dois reservatórios por tubos de diâmetros diferentes. As vazões dependem do fator de atrito, e o fator de atrito depende das vazões. Monte o problema, resolva-o com as ferramentas deste capítulo e verifique a solução. Sugestão: comece com um tubo só e depois acrescente o segundo. O caso com vários tubos vira um **sistema** de equações não lineares, assunto do capítulo 5.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulos 5 e 6 (métodos intervalares e abertos).
- R. P. Brent, *Algorithms for Minimization without Derivatives*, capítulo 4 (o método de Brent).
- A documentação de [`scipy.optimize.root_scalar`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root_scalar.html), que reúne todos os métodos deste capítulo.